# Strat Pred — Stage 4 Diagnostic

Interactive companion to `gcp/research/strat_engine/strat_pred_diagnose.py`. Same
logic, exposed for ad-hoc exploration. Pulls the latest `metrics_*.json`
from GCS for a (ticker, tf) cell and shows:

- Headline metrics (log-loss, accuracy, ECE) + gate verdict
- Per-bin reliability detail (where ECE concentrates, over/under-confidence)
- Per-class P/R/F1 (does the model predict every class?)
- Confusion matrix + predicted-class distribution (the "never predicted" check)

Set `TICKER` and `TF` below, then Run All.

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # so gcp.research... imports

from gcp.research.strat_engine.strat_pred_diagnose import (
    load_metrics, print_summary, print_reliability,
    print_per_class, print_confusion,
)

TICKER = "IWM"
TF = "15m"

## Pull the latest metrics JSON

In [ ]:
m = load_metrics(TICKER, TF)
print_summary(m)

## Reliability (where ECE concentrates)

If a bin shows `← UNDERconfident` repeatedly, the calibrator (likely
isotonic) was too conservative; **sigmoid** often fixes that. If a single
bin is wildly off and others are fine, that bin is hiding behind the
scalar average — fix that bin specifically.

In [ ]:
print_reliability(m)

## Per-class predictability (does the model predict every class?)

If `1` or `3` shows `← NEVER PREDICTED`, the inside/outside half of the
product is broken even though log-loss/accuracy may pass. This is the
class-imbalance failure mode.

In [ ]:
print_per_class(m)

## Confusion + predicted-class distribution

In [ ]:
print_confusion(m)

## Optional: reliability curve as a chart

Skip if you don't have matplotlib available; the table above is the
same data.

In [ ]:
try:
    import matplotlib.pyplot as plt
    bins = [b for b in m['ece_bins'] if b['n'] > 0]
    conf = [b['avg_conf'] for b in bins]
    acc  = [b['avg_acc']  for b in bins]
    sizes = [b['n']/m['n_test']*5000 for b in bins]
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='perfect calibration')
    ax.scatter(conf, acc, s=sizes, alpha=0.7, label='bins (size=weight)')
    for b in bins:
        ax.annotate(f"n={b['n']}", (b['avg_conf'], b['avg_acc']),
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax.set_xlabel('avg predicted probability')
    ax.set_ylabel('empirical accuracy')
    ax.set_title(f"{TICKER} {TF} reliability — ECE={m['ece']:.4f}")
    ax.legend()
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    plt.show()
except ImportError:
    print('matplotlib not available — skipping chart')